# Transfer Learning with ResNet50 on TF Flowers Dataset
This notebook demonstrates transfer learning using ResNet50 to classify the TensorFlow Flowers dataset.

In [14]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os

In [15]:
print("TF version:", tf.__version__)
print("All physical devices:")
print(tf.config.list_physical_devices())

print("\nGPU devices:")
print(tf.config.list_physical_devices("GPU"))

print("Built with GPU support:", tf.test.is_built_with_gpu_support())
print("GPU device name:", tf.test.gpu_device_name())

In [16]:
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
tf.keras.utils.get_file(origin=dataset_url, fname="flower_photos", untar=True)

data_dir = os.path.expanduser("~/.keras/datasets/flower_photos/flower_photos")

print("Data folder:", data_dir)


print(os.listdir(data_dir))

In [17]:
img_size = (224, 224)
batch_size = 128

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int'
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int'
)

In [18]:
class_names = train_ds.class_names
print("Class names:", class_names)

In [19]:
from tensorflow.keras.applications.resnet50 import preprocess_input

def preprocess(image, label):
    image = preprocess_input(image)
    return image, label

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomZoom(0.2),
])  # prevent overfitting

train_ds = train_ds.map(preprocess)
val_ds   = val_ds.map(preprocess)

In [20]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [21]:
def build_model(unfreeze_layers=0):
    base_model = tf.keras.applications.ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )

    base_model.trainable = True     # Freeze all initially
    if unfreeze_layers == 0:
        for layer in base_model.layers:
            layer.trainable = False
    else:
        for layer in base_model.layers[:-unfreeze_layers]:
            layer.trainable = False

    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)
    x = preprocess_input(x)
    x = base_model(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    outputs = tf.keras.layers.Dense(5, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=3e-6,
            weight_decay=1e-5
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [22]:
# 1. Freeze all + 5 epochs
print("Running: Freeze + 5 epochs")
model_f5 = build_model(unfreeze_layers=0)
hist_f5 = model_f5.fit(train_ds, validation_data=val_ds, epochs=5)

# 2. Freeze all + 20 epochs
print("Running: Freeze + 20 epochs")
model_f20 = build_model(unfreeze_layers=0)
hist_f20 = model_f20.fit(train_ds, validation_data=val_ds, epochs=20)

# 3. Unfreeze last 20 layers + 5 epochs
print("Running: Unfreeze 20 + 5 epochs")
model_u5 = build_model(unfreeze_layers=20)
hist_u5 = model_u5.fit(train_ds, validation_data=val_ds, epochs=5)

# 4. Unfreeze last 20 layers + 20 epochs
print("Running: Unfreeze 20 + 20 epochs")
model_u20 = build_model(unfreeze_layers=20)
hist_u20 = model_u20.fit(train_ds, validation_data=val_ds, epochs=20)

In [23]:
histories = {
    "Freeze 5": hist_f5,
    "Freeze 20": hist_f20,
    "Unfreeze 20 - 5": hist_u5,
    "Unfreeze 20 - 20": hist_u20
}

In [24]:
def plot_metric(histories, metric, title=None):
    plt.figure(figsize=(10, 6))

    for name, hist in histories.items():
        train_values = hist.history[metric]
        epochs = range(1, len(train_values) + 1)
        plt.plot(epochs, train_values, label=f"{name} Train {metric}")

        val_metric = f"val_{metric}"
        if val_metric in hist.history:
            val_values = hist.history[val_metric]
            plt.plot(epochs, val_values, '--', label=f"{name} Validation {metric}")

    plt.xlabel("Epoch")
    plt.ylabel(metric.replace('_', ' ').title())

    max_epoch = max(len(h.history.get(metric, [])) for h in histories.values())
    if max_epoch > 0:
        plt.xticks(range(1, max_epoch + 1))

    plt.grid(True)

    if title:
        plt.title(title)
    else:
        plt.title(f"{metric.replace('_', ' ').title()} vs Epoch")

    plt.legend()
    plt.show()


# -------------------------------------------------------------
print("Plotting Accuracy...")
plot_metric(
    histories,
    "accuracy",
    "Comparing Training Strategies: Accuracy vs Epoch"
)

print("Plotting Loss...")
plot_metric(
    histories,
    "loss",
    "Comparing Training Strategies: Loss vs Epoch"
)

In [25]:
def show_predictions(model, dataset, class_names, num_images=9):
    plt.figure(figsize=(12, 12))

    # take a batch
    for images, labels in dataset.take(1):
        preds = model.predict(images)
        preds = np.argmax(preds, axis=1)

        for i in range(num_images):
            plt.subplot(3, 3, i + 1)
            img = images[i].numpy()

            # 從 [0,1] → 顯示
            plt.imshow(img)

            pred_name = class_names[preds[i]]
            true_name = class_names[labels[i]]

            color = "green" if pred_name == true_name else "red"
            plt.title(f"Pred: {pred_name}\nTrue: {true_name}", color=color)
            plt.axis("off")
        break

    plt.tight_layout()
    plt.show()

models = {
    "Freeze 5": model_f5,
    "Freeze 20": model_f20,
    "Unfreeze 20-5": model_u5,
    "Unfreeze 20-20": model_u20
}
for name, m in models.items():
    print(f"=== {name} ===")
    show_predictions(m, val_ds, class_names)